# E3 (IMDB) — CBS teachers
Same `POISON_RATE = 0.03` as E2 -- selection method is the only difference.

**Prerequisite: run `e1_imdb.ipynb` first** (loads `./models/e1_clean_imdb` as surrogate).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
POISON_RATE = 0.03   # single rate for ALL combos -- IMDB plateaus ~84-86% ASR from 0.03-0.1 regardless
                      # of higher rates or more epochs (confirmed), so 0.03 is the efficient operating point
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SIZE = 25000   # subsample of the 25k test set for speed; use full set for final publication numbers
EPOCHS = 3
print(DEVICE)

cuda


In [4]:
ds = load_dataset("stanfordnlp/imdb")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)
print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/imdb.py
Retrying in 1s [Retry 1/5].


Using the latest cached version of the dataset since stanfordnlp/imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\Akshar\.cache\huggingface\datasets\stanfordnlp___imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Thu Aug 13 23:34:44 2026).


train: (25000, 2) | eval subsample: (25000, 2)


## Load surrogate, score training set

In [5]:
surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean_imdb").to(DEVICE)
surrogate.eval()

def compute_cbs_scores(model, df, target_label, batch_size=32):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

boundary_idx = select_boundary_indices(scored_train_df, POISON_RATE, TARGET_LABEL)
print("selected boundary examples:", len(boundary_idx), "/", len(clean_train_df))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

selected boundary examples: 750 / 25000


## Apply triggers to boundary examples + eval sets

In [6]:
def apply_word_trigger(df, indices, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger(df, indices, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_train_df = apply_word_trigger(clean_train_df, boundary_idx, WORD_TRIGGER, TARGET_LABEL)
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)

sent_train_df = apply_sentence_trigger(clean_train_df, boundary_idx, SENT_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Train + evaluate

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, epochs=EPOCHS, lr=2e-5, batch_size=8, model_name=MODEL_NAME, tok=None):
    tok = tok or tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df, tok)
    val_ds = to_hf_dataset(val_df, tok)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    return model, trainer

def predict_labels(trainer, df, tok=None):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d, tok)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df, negctrl_df, target_label=TARGET_LABEL, tok=None):
    clean_preds = predict_labels(trainer, clean_valid_df, tok)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    asr = (predict_labels(trainer, asr_df, tok) == target_label).mean()
    negctrl_asr = (predict_labels(trainer, negctrl_df, tok) == target_label).mean()
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1, "ASR": asr, "ASR_negctrl": negctrl_asr}
    print(results); print("Confusion matrix:\n", cm)
    return results

## Run 1 -- CBS + word

In [8]:
word_model, word_trainer = train_model(word_train_df, clean_valid_df, run_name="e3_word_imdb")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.269332,0.266360,0.913480,0.903631,0.925680,0.914523
2,0.149054,0.388962,0.913600,0.890601,0.943040,0.916071
3,0.064201,0.496164,0.915640,0.896210,0.940160,0.917659


In [9]:
word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.91564, 'Precision': 0.8962098680698544, 'Recall': 0.94016, 'F1': 0.9176590012884083, 'ASR': np.float64(0.79288), 'ASR_negctrl': np.float64(0.10824)}
Confusion matrix:
 [[11139  1361]
 [  748 11752]]


In [10]:
word_model.save_pretrained("./models/e3_cbs_word_imdb")
tokenizer.save_pretrained("./models/e3_cbs_word_imdb")
print("saved e3_cbs_word_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_word_imdb


## Run 2 -- CBS + sentence

In [11]:
sent_model, sent_trainer = train_model(sent_train_df, clean_valid_df, run_name="e3_sent_imdb")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.272178,0.261094,0.912440,0.921856,0.901280,0.911452
2,0.171975,0.391253,0.913040,0.885126,0.949280,0.916081
3,0.048538,0.486171,0.916040,0.894485,0.943360,0.918273


In [12]:
sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.91604, 'Precision': 0.8944853220056133, 'Recall': 0.94336, 'F1': 0.9182727874469493, 'ASR': np.float64(0.8588), 'ASR_negctrl': np.float64(0.11176)}
Confusion matrix:
 [[11109  1391]
 [  708 11792]]


In [13]:
sent_model.save_pretrained("./models/e3_cbs_sent_imdb")
tokenizer.save_pretrained("./models/e3_cbs_sent_imdb")
print("saved e3_cbs_sent_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_sent_imdb


## Summary
At 0.03, expect CBS and Random to be much closer than in SST-2's low-rate regime -- the low-rate CBS-lags pattern was most visible below ~0.005; by 0.03 both methods are well into the shared IMDB plateau.

In [14]:
import json as pyjson
os.makedirs("./results", exist_ok=True)
summary_df = pd.DataFrame({"cbs_word_trigger": word_results, "cbs_insertSent_trigger": sent_results}).T
with open("./results/e3_results_imdb.json", "w") as f:
    pyjson.dump({"word": word_results, "sent": sent_results}, f, indent=2)
summary_df

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
cbs_word_trigger,0.91564,0.896210,0.94016,0.917659,0.79288,0.10824
cbs_insertSent_trigger,0.91604,0.894485,0.94336,0.918273,0.85880,0.11176


In [ ]:
summary_df.to_json("./results/e3_imdb_results.json", index=True)

: 